In [1]:
%load_ext autoreload
%autoreload 2

# Sankey example

In [2]:
import sys
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
# Define the nodes
nodes = [
    {'label': 'Node 1'},
    {'label': 'Node 2'},
    {'label': 'Node 3'},
    {'label': 'Node 4'},
    {'label': 'Node 5'},
    {'label': 'Node 6'},
    {'label': 'Node 7'}, 
    {'label': 'Node 8'}
]

# Define the links
links = [
    {'source': 0, 'target': 1, 'value': 10},  # Flow from Node 1 to Node 2 with a value of 10
    {'source': 1, 'target': 2, 'value': 5},   # Flow from Node 2 to Node 3 with a value of 5
    {'source': 2, 'target': 3, 'value': 1},   # Flow from Node 3 to Node 1 with a value of 1
    {'source': 4, 'target': 2, 'value': 2},   # Flow from Node 4 to Node 1 with a value of 2
    {'source': 5, 'target': 1, 'value': 8},   # Flow from Node 5 to Node 1 with a value of 8
    {'source': 6, 'target': 3, 'value': 3},   # Flow from Node 6 to Node 1 with a value of 3
    {'source': 6, 'target': 1, 'value': 7},   # Flow from Node 7 to Node 1 with a value of 7
    {'source': 1, 'target': 7, 'value': 4},   # Flow from Node 8 to Node 1 with a value of 4
    {'source': 7, 'target': 3, 'value': 6},   # Flow from Node 9 to Node 1 with a value of 6
]
type(links[0])
print(len(nodes))

8


In [4]:
# Create the figure and subplot
fig = make_subplots(rows=1, cols=1)

# Add the Sankey trace
fig.add_trace(go.Sankey(
    node=dict(
        pad=10*len(nodes),  # Adjust the padding between nodes
        thickness=20,  # Adjust the thickness of nodes
        line=dict(color='black', width=1),  # Define the color and width of node lines
        label=[node['label'] for node in nodes],  # Extract node labels from the 'nodes' list
    ),
    link=dict(
        source=[link['source'] for link in links],  # Extract source indices from the 'links' list
        target=[link['target'] for link in links],  # Extract target indices from the 'links' list
        value=[link['value'] for link in links],  # Extract values from the 'links' list
    )
))

# Update the layout
fig.update_layout(title_text='Sankey Diagram')

# Show the plot
fig.show()


# Sankey dataframe

In [5]:
from sklearn.cluster import KMeans
import numpy as np

In [6]:
import numpy as np
from sklearn.cluster import KMeans

def detect_label_switching(
        data : np.ndarray,
        num_clusters_range: range
        ) -> np.ndarray:
    """
    Detects label switching in KMeans clustering from one iteration to the next.

    Parameters
    ----------
    data : numpy.ndarray
        One-dimensional array of data points to cluster.
    num_clusters_range : range
        Range of number of clusters to try.
    
    Returns
    -------
    numpy.ndarray
        Array of labels for each data point for each iteration of KMeans clustering.
    """
    prev_labels = None  # Initialize previous labels as None
    km_labels = np.zeros((data.shape[0],data.shape[0]), dtype=int)
    km_labels_memorize = np.zeros((data.shape[0],data.shape[0]), dtype=int)

    for num_clusters in num_clusters_range:
        kmeans = KMeans(n_clusters=data.shape[0] - num_clusters, random_state=0, n_init=10)
        kmeans.fit(data.reshape(-1, 1))
        labels = kmeans.labels_
        km_labels[num_clusters] = labels
        if num_clusters == 0:
            km_labels_memorize[num_clusters] = labels
        print('previous iteration', prev_labels)
        if prev_labels is not None:
            print('num_cluster is', num_clusters)
            # Compare current labels with previous labels
            switched_indices = np.where(labels != prev_labels)[0]
            print('switched indices', switched_indices)
            if len(switched_indices) > 0:
                #print(f"Label switching detected at iteration {num_clusters - 1} to {num_clusters}:")
                for index in switched_indices:
                    prev_label = prev_labels[index]
                    curr_label = labels[index]
                    #print(f"Data point at index {index}: switched from label {prev_label} to label {curr_label}")
            km_labels_memorize[num_clusters] = km_labels_memorize[num_clusters - 1]
            print('before changing previous iteration', km_labels_memorize[num_clusters])
            for i, index in enumerate(switched_indices):
                print('subiteration is', labels[index])
                print('TEST', np.where(km_labels_memorize[num_clusters-1] == km_labels_memorize[num_clusters-1][index]))
                print(np.where(km_labels_memorize[num_clusters-1] == km_labels_memorize[num_clusters-1][index])[0])
                num_repeated_labels = len(np.where(km_labels_memorize[num_clusters-1] == km_labels_memorize[num_clusters-1][index])[0])
                print('give number of repeated label', num_repeated_labels)
                if  num_repeated_labels>1 and prev_labels is not None and num_clusters>0:
                    print("here")
                    all_index = np.where(km_labels[num_clusters] == labels[index])[0]
                    km_labels_memorize[num_clusters][all_index] = int(np.max(km_labels_memorize[num_clusters - 1])) + i + 1
        
            
        prev_labels = labels
    return km_labels, km_labels_memorize

# Example usage:
data = np.array([0.59830198, 0.7101328 , 0.25210179, 0.90204906, 0.79986231, 0.25293537]) #0.21397742, 0.50635925, 0.67623288, 0.74856172, 
num_clusters_range = range(data.shape[0])  # Example range of number of clusters

result, result_new_labels = detect_label_switching(data, num_clusters_range)


previous iteration None
previous iteration [2 3 5 4 0 1]
num_cluster is 1
switched indices [2]
before changing previous iteration [2 3 5 4 0 1]
subiteration is 1
TEST (array([2], dtype=int64),)
[2]
give number of repeated label 1
previous iteration [2 3 1 4 0 1]
num_cluster is 2
switched indices [0 1 3 4]
before changing previous iteration [2 3 5 4 0 1]
subiteration is 0
TEST (array([0], dtype=int64),)
[0]
give number of repeated label 1
subiteration is 2
TEST (array([1], dtype=int64),)
[1]
give number of repeated label 1
subiteration is 3
TEST (array([3], dtype=int64),)
[3]
give number of repeated label 1
subiteration is 2
TEST (array([4], dtype=int64),)
[4]
give number of repeated label 1
previous iteration [0 2 1 3 2 1]
num_cluster is 3
switched indices [0 3 4]
before changing previous iteration [2 3 5 4 0 1]
subiteration is 2
TEST (array([0], dtype=int64),)
[0]
give number of repeated label 1
subiteration is 0
TEST (array([3], dtype=int64),)
[3]
give number of repeated label 1
subi

In [7]:
print(data)
print(result)
print(result_new_labels)

[0.59830198 0.7101328  0.25210179 0.90204906 0.79986231 0.25293537]
[[2 3 5 4 0 1]
 [2 3 1 4 0 1]
 [0 2 1 3 2 1]
 [2 2 1 0 0 1]
 [0 0 1 0 0 1]
 [0 0 0 0 0 0]]
[[2 3 5 4 0 1]
 [2 3 5 4 0 1]
 [2 3 5 4 0 1]
 [2 3 5 4 0 1]
 [2 3 5 4 0 1]
 [2 3 5 4 0 1]]


In [8]:
import numpy as np
import plotly.graph_objects as go
from sklearn.cluster import KMeans

def visualize_label_switching(data, num_clusters_range):
    prev_labels = None  # Initialize previous labels as None
    links = []

    for num_clusters in num_clusters_range:
        kmeans = KMeans(n_clusters=num_clusters)
        kmeans.fit(data.reshape(-1, 1))
        labels = kmeans.labels_

        if prev_labels is not None:
            # Compare current labels with previous labels
            switched_indices = np.where(labels != prev_labels)[0]
            if len(switched_indices) > 0:
                for index in switched_indices:
                    prev_label = prev_labels[index]
                    curr_label = labels[index]
                    links.append({"source": prev_label, "target": curr_label, "value": 1})

        prev_labels = labels

    # Create nodes
    nodes = [{"label": label} for label in np.unique(prev_labels)]

    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(node=dict(label=node["label"]), link=dict(source=links["source"], target=links["target"], value=links["value"])) for node in nodes], layout_title_text="Label Switching Sankey Diagram")

    # Show the Sankey diagram
    fig.show()

# Example usage:
data = np.array([2, 4, 6, 10, 12, 15, 18, 20, 25, 30])
num_clusters_range = range(2, 6)  # Example range of number of clusters

visualize_label_switching(data, num_clusters_range)


c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning

c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning

c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning

c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning



TypeError: list indices must be integers or slices, not str

# Deterministic Sankey from method

In [9]:
# Generate sbm and adjacency matrix
import sys
sys.path.append('..')
from pygraphon.graphons import Graphon
from pygraphon.graphons import StepGraphon
from experiments.utils_generators import AssociativeFullyRandom
k = 2
n = 40
# theta init
theta_intra_min, theta_intra_max = 0.5, 0.7
theta_inter_min, theta_inter_max = 0.1, 0.4
theta = AssociativeFullyRandom(
                    K=k,
                    min_inside_classes=theta_intra_min,
                    max_inside_classes=theta_intra_max,
                    min_outside_classes=theta_inter_min,
                    max_outside_classes=theta_inter_max,
                )
graphon = StepGraphon(theta.Theta, bandwidthHist=1 / k)
adjacency_matrix=graphon.draw(n=n, exchangeable=False)

In [10]:
#import smoothnethist
from pygraphon.estimators import SmoothNetHist

est = SmoothNetHist(bandwidth=1/(2*k))
est.fit(adjacency_matrix)

In [46]:
list_graphons = [est.all_cluster_graphons[i].graphon[np.triu_indices(est.all_cluster_graphons[i].graphon.shape[0])] for i in range(len(est.all_cluster_graphons))]

sort_indices = np.argsort(list_graphons[-1])

for i in range(len(list_graphons)):
    list_graphons[i] = list_graphons[i][sort_indices]
graphons_array = np.array(list_graphons)
graphons_array = np.rot90(np.insert(graphons_array, 0, np.ones(graphons_array.shape[1]), axis=0), k=3)

In [53]:
list_graphons

[array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]),
 array([0.60112507, 0.60112507, 0.60112507, 0.60112507, 0.60112507,
        1.5128392 , 1.5128392 , 1.5128392 , 1.5128392 , 1.5128392 ]),
 array([0.60488034, 0.60488034, 0.60488034, 0.60488034, 0.60488034,
        1.22243078, 1.22243078, 1.72219615, 1.72219615, 1.72219615]),
 array([0.25480149, 0.69657537, 0.69657537, 0.69657537, 0.69657537,
        1.22918126, 1.22918126, 1.73170642, 1.73170642, 1.73170642]),
 array([0.25802598, 0.63789757, 0.63789757, 0.63789757, 0.9078692 ,
        1.24473645, 1.24473645, 1.75362102, 1.75362102, 1.75362102]),
 array([0.25891519, 0.64009589, 0.64009589, 0.64009589, 0.9109979 ,
        1.24902607, 1.24902607, 1.70452502, 1.70452502, 1.86994306]),
 array([0.25891519, 0.58255918, 0.66886425, 0.66886425, 0.9109979 ,
        1.24902607, 1.24902607, 1.70452502, 1.70452502, 1.86994306]),
 array([0.25926482, 0.58334584, 0.66976744, 0.66976744, 0.91222806,
        1.25071268, 1.25071268, 1.68522131, 1.72843211

In [52]:
graphons_array

array([[0.25922592, 0.25926482, 0.25891519, 0.25891519, 0.25802598,
        0.25480149, 0.60488034, 0.60112507, 1.        , 1.        ],
       [0.58325833, 0.58334584, 0.58255918, 0.64009589, 0.63789757,
        0.69657537, 0.60488034, 0.60112507, 1.        , 1.        ],
       [0.66966697, 0.66976744, 0.66886425, 0.64009589, 0.63789757,
        0.69657537, 0.60488034, 0.60112507, 1.        , 1.        ],
       [0.66966697, 0.66976744, 0.66886425, 0.64009589, 0.63789757,
        0.69657537, 0.60488034, 0.60112507, 1.        , 1.        ],
       [0.91209121, 0.91222806, 0.9109979 , 0.9109979 , 0.9078692 ,
        0.69657537, 0.60488034, 0.60112507, 1.        , 1.        ],
       [1.24812481, 1.25071268, 1.24902607, 1.24902607, 1.24473645,
        1.22918126, 1.22243078, 1.5128392 , 1.        , 1.        ],
       [1.25292529, 1.25071268, 1.24902607, 1.24902607, 1.24473645,
        1.22918126, 1.22243078, 1.5128392 , 1.        , 1.        ],
       [1.6849685 , 1.68522131, 1.7045250

In [51]:
#makes list into pandas dataframe
import pandas as pd
pd.set_option('display.float_format', '{:.3f}'.format)
columns = []
for i in range(len(list_graphons)):
    columns.append("SmoothNetHist" + str(i))
columns.append("Count")

def list_to_df(array, columns):
    df = pd.DataFrame(array, columns=columns)
    return df

df = list_to_df(graphons_array, columns=columns)
df

,SmoothNetHist0,SmoothNetHist1,SmoothNetHist2,SmoothNetHist3,SmoothNetHist4,SmoothNetHist5,SmoothNetHist6,SmoothNetHist7,SmoothNetHist8,Count
0,0.259,0.259,0.259,0.259,0.258,0.255,0.605,0.601,1.000,1.000
1,0.583,0.583,0.583,0.640,0.638,0.697,0.605,0.601,1.000,1.000
2,0.670,0.670,0.669,0.640,0.638,0.697,0.605,0.601,1.000,1.000
3,0.670,0.670,0.669,0.640,0.638,0.697,0.605,0.601,1.000,1.000
4,0.912,0.912,0.911,0.911,0.908,0.697,0.605,0.601,1.000,1.000
5,1.248,1.251,1.249,1.249,1.245,1.229,1.222,1.513,1.000,1.000
6,1.253,1.251,1.249,1.249,1.245,1.229,1.222,1.513,1.000,1.000
7,1.685,1.685,1.705,1.705,1.754,1.732,1.722,1.513,1.000,1.000
8,1.728,1.728,1.705,1.705,1.754,1.732,1.722,1.513,1.000,1.000
9,1.872,1.872,1.870,1.870,1.754,1.732,1.722,1.513,1.000,1.000
